In [ ]:
import topology
from tqdm import tqdm
import numpy as np
import matplotlib.pyplot as plt
import territorial_automaton as ta
import visualization as vis

# grid, pos = topology.grid(50, periodic=False)
# topology.plot_graph(grid)

adj, pos = topology.load_graphml('test_topologies/galaxy_graph_5.graphml')
topology.plot_graph(adj, pos)

In [ ]:
initial = np.random.choice(ta.STATES, size=adj.shape[0])
params = ta.TA_Params(adj, T=1.0, h=0.0, theta=0.8, kappa=-0.5, initial_state=initial)
model = ta.TerritorialAutomaton(params)
metrics = model.run(n_warmup=0, n_experiment=1000, snapshots=True, progbar=True).metrics
print(f'Final order: {metrics[-1].order:.4f}, Final energy: {metrics[-1].energy:.4f}')
vis.animate_simulation(metrics, model.params.adj_matrix, pos, interval=1, node_size=50, save_path='sw.mp4')

In [ ]:
runs = 100
orders = np.zeros(runs)

model = ta.TerritorialAutomaton(ta.TA_Params(adj, T=0.5, h=0.0, theta=0.5, kappa=-0.8))
for i in tqdm(range(runs)):
    metrics = model.run(n_warmup=0, n_experiment=500).metrics
    orders[i] = metrics[-1].order

print(f'Average order over {runs} runs: {np.mean(orders):.4f}')
print(f'Standard error of the mean: {np.std(orders, ddof=1)/np.sqrt(runs):.4f}')

In [ ]:
# plot orders histogram
plt.hist(orders, bins=20, density=True, alpha=0.7)
plt.xlabel('Order')
plt.ylabel('Density')
plt.title('Distribution of Final Order Values')
plt.savefig('order_histogram.png')
plt.show()

In [ ]:
steps = 3000
max_T = 4.0
steps_per_T = 10

params = ta.TA_Params(adj, T=max_T, h=0.0, theta=0.8, kappa=-0.5)
model = ta.TerritorialAutomaton(params)
dynamic_T = np.linspace(0, max_T, steps//(2 * steps_per_T))
dynamic_T = np.repeat(dynamic_T, steps_per_T)  # Hold each T value for steps_per_T steps
dynamic_T = np.concatenate([dynamic_T[::-1], dynamic_T])  # Create a triangular wave for T
print(dynamic_T.shape)
metrics = model.run(n_warmup=10, n_experiment=steps, dynamic_T=dynamic_T, snapshots=True, progbar=True).metrics

print(f'Order at t=500: {metrics[500].order:.4f}, Energy at t=500: {metrics[500].energy:.4f}')

In [ ]:
vis.animate_simulation(metrics, model.params.adj_matrix, pos, interval=1, node_size=50, save_path='dynamic_T.gif')

orders = [m.order for m in metrics]
energies = [m.energy for m in metrics]
# plot order and temperature over time with shared x-axis
fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(8, 12), sharex=True)
fig.suptitle('Order Parameter and Temperature Over Time with Dynamic Temperature', fontsize=16)
ax1.plot(orders, label='Order Parameter')
ax1.set_ylabel('Order')
ax1.set_title('Order Parameter Over Time with Dynamic Temperature')
ax2.plot(energies, label='Energy')
ax2.set_ylabel('Energy')
ax2.set_title('Energy Over Time')
ax3.plot(dynamic_T, label='Temperature (T)', alpha=0.5)
ax3.set_ylabel('Temperature')
ax3.set_title('Temperature Over Time')

plt.xlabel('Time Step')
plt.legend()
plt.grid()
plt.savefig('dynamic_T_order.png')